In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import re
import json
import random
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed,
)

MODEL_NAME = "HooshvareLab/bert-base-parsbert-uncased"

FACTUAL_PATH = (
    "/kaggle/input/notebooks/aabdollahii/"
    "verbalizer/kg_sentences_all_clean.txt"
)

GENERAL_PATH = (
    "/kaggle/input/datasets/miladfa7/"
    "persian-wikipedia-dataset/Persian-WikiText-1.txt"
)

OUTPUT_DIR = "/kaggle/working/parsbert_kg_mlm"

SEED = 42
MAX_LENGTH = 128
MLM_PROB = 0.15

FACTUAL_RATIO = 0.8
MIN_LEN_CHARS = 3

MAX_FACTUAL_LINES = 800_000
MAX_GENERAL_LINES = 200_000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def basic_cleanup(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\[https?://[^\]]+\]", "", text)
    text = re.sub(r"<ref[^>]*>.*?</ref>", "", text)
    text = re.sub(r"<ref[^/]*/>", "", text)
    return text.strip()


def simple_normalize(text: str) -> str:
    text = basic_cleanup(text)
    text = text.replace("\u064A", "\u06CC")
    text = text.replace("\u0643", "\u06A9")
    return text.strip()


def load_lines(path: str, limit: int = None):
    if path is None or not os.path.exists(path):
        return []

    lines = []

    with open(path, "r", encoding="utf-8") as file:
        for index, line in enumerate(file):
            if limit is not None and index >= limit:
                break

            text = simple_normalize(line)

            if len(text) >= MIN_LEN_CHARS:
                lines.append(text)

    return lines


factual_lines = load_lines(
    FACTUAL_PATH,
    limit=MAX_FACTUAL_LINES,
)

general_lines = load_lines(
    GENERAL_PATH,
    limit=MAX_GENERAL_LINES,
)

if len(factual_lines) == 0:
    raise ValueError(
        "No factual lines loaded. Check FACTUAL_PATH."
    )

if general_lines:
    n_total = len(factual_lines) + len(general_lines)

    n_factual_target = int(FACTUAL_RATIO * n_total)
    n_general_target = n_total - n_factual_target

    factual_sample = (
        factual_lines
        if len(factual_lines) <= n_factual_target
        else random.sample(factual_lines, n_factual_target)
    )

    general_sample = (
        general_lines
        if len(general_lines) <= n_general_target
        else random.sample(general_lines, n_general_target)
    )

    all_lines = factual_sample + general_sample
    random.shuffle(all_lines)
else:
    all_lines = factual_lines.copy()
    random.shuffle(all_lines)

print("Number of factual lines:", len(factual_lines))
print("Number of general lines:", len(general_lines))
print("Total training lines:", len(all_lines))

train_dataset = Dataset.from_dict({
    "text": all_lines
})

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_special_tokens_mask=True,
    )


tokenized_dataset = train_dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=1000,
    remove_columns=["text"],
    desc="Tokenizing training dataset",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROB,
)

model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_steps=500,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    lr_scheduler_type="linear",
    fp16=torch.cuda.is_available(),
    logging_strategy="steps",
    logging_steps=200,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

train_result = trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print("Training completed.")
print("Final model:", OUTPUT_DIR)
print("Training metrics:", train_result.metrics)
